# Train TRRUST Classifiers

Hyperparameter tuning with 5-fold cross-validation per config, optional sample
subsetting, and final-results cross-validation for binary (Relationship vs None)
and ternary (Activation vs Repression vs None) TRRUST classifiers.

Pipeline:

1. **Load** gene embeddings + TRRUST data.
2. **(Optional)** Restrict training data to a gene subset from a file.
3. **(Optional)** Write the current gene universe to a file for cross-model comparisons.
4. **Hyperparameter sweep with 5-fold CV per config** — 5 LRs × 4 epoch counts
   for binary + ternary. For each config we run stratified 5-fold CV and save
   per-fold train/test loss curves, per-fold classification reports, a summary
   CSV with mean/std accuracy and macro F1, and a heatmap of **mean macro F1**
   across LR × epochs under `REPORTS_DIR`.
5. **Final results** — re-run 5-fold CV at the chosen LR / epoch count for
   binary + ternary (inspect the sweep heatmap, then set `*_LR` / `*_EPOCHS`).
6. **Save final results** as a pickle for downstream analysis.

This notebook is model-agnostic — point `EMBEDDINGS_PATH` at any h5ad file produced
by `src/scgpt/encode.py` or `src/geneformer/encode.py`.


In [ ]:
import itertools
import pickle
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report

from src.trrust import (
    BINARY_LABEL_NAMES,
    BINARY_LABELS,
    TERNARY_LABEL_NAMES,
    TERNARY_LABELS,
    cross_validate,
    filter_data_by_genes,
    load_binary_trrust_data,
    load_gene_embeddings,
    load_ternary_trrust_data,
)


## Configuration

Change `EMBEDDINGS_PATH` to swap foundation models, and point `REPORTS_DIR` /
`CV_RESULTS_PATH` at model-specific output locations. Set `GENE_SUBSET_PATH` to
restrict training to pairs where both TF and target appear in a gene-list file
(see `data/scgpt_binary_genes.txt`).

`LR_GRID` and `EPOCH_GRID` define the hyperparameter sweep; each `(lr, epochs)`
config is evaluated with 5-fold stratified CV. After inspecting the mean macro
F1 heatmap, fill in `BINARY_LR` / `BINARY_EPOCHS` / `TERNARY_LR` /
`TERNARY_EPOCHS` before running the final-results cells.


In [ ]:
EMBEDDINGS_PATH = Path("../data/embeddings/scgpt_human_cd20.h5ad")
TRRUST_PATH = Path("../data/trrust_rawdata.human.tsv")
REPORTS_DIR = Path("../reports/scgpt/hp_tuning_cd20")
CV_RESULTS_PATH = Path("../data/scgpt_cv_results_cd20.pkl")
GENE_SUBSET_PATH: Path | None = None  # e.g. Path("../data/scgpt_binary_genes.txt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

LR_GRID = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
EPOCH_GRID = [6, 12, 20, 50]


print(f"Device: {DEVICE}")
print(f"Embeddings: {EMBEDDINGS_PATH}")
print(f"TRRUST: {TRRUST_PATH}")
print(f"Reports dir: {REPORTS_DIR}")
print(f"CV results: {CV_RESULTS_PATH}")
print(f"Gene subset: {GENE_SUBSET_PATH}")


Device: cuda
Embeddings: ../data/embeddings/scgpt_human_cd20.h5ad
TRRUST: ../data/trrust_rawdata.human.tsv
Reports dir: ../reports/scgpt/hp_tuning_cd20
CV results: ../data/scgpt_cv_results_cd20.pkl
Gene subset: None


## Load embeddings and TRRUST data

In [5]:
gene_embeddings = load_gene_embeddings(EMBEDDINGS_PATH)
embsize = next(iter(gene_embeddings.values())).shape[0]

binary_data = load_binary_trrust_data(TRRUST_PATH, gene_embeddings)
ternary_data = load_ternary_trrust_data(TRRUST_PATH, gene_embeddings)

print(f"Gene embeddings: {len(gene_embeddings)} genes, {embsize}d")
print(f"Binary samples: {len(binary_data.records)}")
print(f"Ternary samples: {len(ternary_data.records)}")


Gene embeddings: 10856 genes, 512d
Binary samples: 8100
Ternary samples: 3147


## (Optional) Restrict training data to a gene subset

If `GENE_SUBSET_PATH` is set, keep only TRRUST records where both TF and target are
listed in the file (one gene symbol per line). A no-op when `GENE_SUBSET_PATH is None`.


In [ ]:
if GENE_SUBSET_PATH is not None:
    subset_genes = {
        line.strip()
        for line in Path(GENE_SUBSET_PATH).read_text().splitlines()
        if line.strip()
    }
    print(f"Subset file: {GENE_SUBSET_PATH} ({len(subset_genes)} genes)")

    before_binary = len(binary_data.records)
    before_ternary = len(ternary_data.records)
    binary_data = filter_data_by_genes(binary_data, subset_genes)
    ternary_data = filter_data_by_genes(ternary_data, subset_genes)
    print(f"Binary records: {before_binary} -> {len(binary_data.records)}")
    print(f"Ternary records: {before_ternary} -> {len(ternary_data.records)}")
else:
    print("GENE_SUBSET_PATH is None — using all records.")


## (Optional) Save current gene lists for future cross-model runs

Write the union of TFs + targets from the current `binary_data` / `ternary_data` to
two `.txt` files so another run (e.g. a different embedding model) can be restricted
to the same gene universe via `GENE_SUBSET_PATH` above. Edit the output paths for
the model you are running.


In [ ]:
BINARY_GENE_FILE = Path("../data/scgpt_binary_genes.txt")
TERNARY_GENE_FILE = Path("../data/scgpt_ternary_genes.txt")

binary_genes = sorted({g for r in binary_data.records for g in (r.tf, r.target)})
BINARY_GENE_FILE.write_text("\n".join(binary_genes) + "\n")
print(f"Saved {len(binary_genes)} binary genes to {BINARY_GENE_FILE}")

ternary_genes = sorted({g for r in ternary_data.records for g in (r.tf, r.target)})
TERNARY_GENE_FILE.write_text("\n".join(ternary_genes) + "\n")
print(f"Saved {len(ternary_genes)} ternary genes to {TERNARY_GENE_FILE}")


## Hyperparameter sweep helpers

Each config runs 5-fold stratified cross-validation (seed=42) so rankings are
robust to any single split. For every `(lr, epochs)` combination we save a
single plot with overlaid per-fold train/test loss curves, a text report with
per-fold classification reports plus mean ± std accuracy / macro F1, and a row
in `summary.csv`. After the loop, a heatmap of **mean macro F1** over LR ×
epochs is saved alongside the per-config outputs.


In [ ]:
def _save_loss_plot(cv_result, lr, epochs, title_prefix, out_path):
    plt.figure(figsize=(8, 4))
    xs = range(1, epochs + 1)
    colors = plt.get_cmap("tab10").colors
    for i, fold in enumerate(cv_result.per_fold):
        color = colors[i % len(colors)]
        plt.plot(xs, fold.train_losses, color=color, linestyle="-",
                 label=f"Fold {i + 1} train")
        plt.plot(xs, fold.test_losses, color=color, linestyle="--",
                 label=f"Fold {i + 1} test")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} — lr={lr:.0e}, epochs={epochs}")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def _fold_macro_f1s(cv_result):
    return np.array([
        fold.classification_report["macro avg"]["f1-score"]
        for fold in cv_result.per_fold
    ])


def _save_text_report(cv_result, label_names, out_path):
    target_names = [label_names[i] for i in range(len(label_names))]
    name_to_label = {name: idx for idx, name in label_names.items()}
    lines = []
    for i, fold in enumerate(cv_result.per_fold):
        preds = fold.gene_predictions
        true_ids = preds["true_relationship"].map(name_to_label).to_numpy()
        pred_ids = preds["predicted_relationship"].map(name_to_label).to_numpy()
        lines.append(f"=== Fold {i + 1} ===")
        lines.append(classification_report(
            true_ids, pred_ids, target_names=target_names, zero_division=0
        ))
    accs = np.array(cv_result.fold_accuracies)
    f1s = _fold_macro_f1s(cv_result)
    lines.append(f"Mean accuracy: {accs.mean():.3f} ± {accs.std():.3f}")
    lines.append(f"Mean macro F1: {f1s.mean():.3f} ± {f1s.std():.3f}")
    out_path.write_text("\n".join(lines))


def _summary_row(lr, epochs, cv_result, label_names):
    accs = np.array(cv_result.fold_accuracies)
    f1s = _fold_macro_f1s(cv_result)
    row = {
        "lr": lr,
        "epochs": epochs,
        "mean_accuracy": accs.mean(),
        "std_accuracy": accs.std(),
        "mean_macro_f1": f1s.mean(),
        "std_macro_f1": f1s.std(),
    }
    agg = cv_result.aggregate_classification_report
    row["pooled_accuracy"] = agg["accuracy"]
    for name in ("macro avg", "weighted avg"):
        for metric in ("precision", "recall", "f1-score"):
            row[f"pooled_{name}_{metric}"] = agg[name][metric]
    for i in range(len(label_names)):
        name = label_names[i]
        row[f"pooled_{name}_precision"] = agg[name]["precision"]
        row[f"pooled_{name}_recall"] = agg[name]["recall"]
        row[f"pooled_{name}_f1"] = agg[name]["f1-score"]
    return row


def _save_f1_heatmap(summary_df, title, out_path):
    pivot = summary_df.pivot(
        index="lr", columns="epochs", values="mean_macro_f1"
    ).sort_index(ascending=False)
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.0e}" for v in pivot.index])
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Learning rate")
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center",
                    color="white" if pivot.values[i, j] < pivot.values.mean() else "black")
    fig.colorbar(im, ax=ax, label="mean macro F1")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def run_hp_sweep(
    data,
    *,
    label_map,
    label_names,
    use_class_weights,
    out_dir,
    title_prefix,
    n_splits=5,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for lr, epochs in itertools.product(LR_GRID, EPOCH_GRID):
        cv_result = cross_validate(
            data,
            embsize=embsize,
            label_map=label_map,
            lr=lr,
            epochs=epochs,
            batch_size=BATCH_SIZE,
            use_class_weights=use_class_weights,
            n_splits=n_splits,
            device=DEVICE,
            seed=42,
        )
        tag = f"lr{lr:.0e}_e{epochs}"
        _save_loss_plot(cv_result, lr, epochs, title_prefix, out_dir / f"loss_{tag}.png")
        _save_text_report(cv_result, label_names, out_dir / f"report_{tag}.txt")
        row = _summary_row(lr, epochs, cv_result, label_names)
        rows.append(row)
        print(f"  {tag}: mean acc={row['mean_accuracy']:.3f} ± {row['std_accuracy']:.3f}, "
              f"mean macro F1={row['mean_macro_f1']:.3f} ± {row['std_macro_f1']:.3f}")

    summary = pd.DataFrame(rows)
    summary.to_csv(out_dir / "summary.csv", index=False)
    _save_f1_heatmap(
        summary,
        f"{title_prefix} mean macro F1 (5-fold CV)",
        out_dir / "f1_heatmap.png",
    )
    return summary


## Hyperparameter sweep — binary classifier

20 configs (5 LRs × 4 epoch counts), each run through 5-fold stratified CV.
No class weights (binary data is balanced by construction). Outputs land in
`REPORTS_DIR/binary/`.


In [ ]:
binary_summary = run_hp_sweep(
    binary_data,
    label_map=BINARY_LABELS,
    label_names=BINARY_LABEL_NAMES,
    use_class_weights=False,
    out_dir=REPORTS_DIR / "binary",
    title_prefix="Binary",
)
binary_summary.sort_values("mean_macro_f1", ascending=False).head()


## Hyperparameter sweep — ternary classifier

20 configs, each run through 5-fold stratified CV with inverse-frequency class
weights (ternary data is imbalanced). Outputs land in `REPORTS_DIR/ternary/`.


In [ ]:
ternary_summary = run_hp_sweep(
    ternary_data,
    label_map=TERNARY_LABELS,
    label_names=TERNARY_LABEL_NAMES,
    use_class_weights=True,
    out_dir=REPORTS_DIR / "ternary",
    title_prefix="Ternary",
)
ternary_summary.sort_values("mean_macro_f1", ascending=False).head()


## Final results — binary classifier

Re-run 5-fold CV at the chosen `(BINARY_LR, BINARY_EPOCHS)` (read off the sweep
heatmap) to produce the final reportable per-fold accuracies and the pooled
(aggregate) classification report.


In [13]:
BINARY_LR = 0.001
BINARY_EPOCHS = 20
BINARY_FOLDS = 5

In [14]:
binary_cv = cross_validate(
    binary_data,
    embsize=embsize,
    label_map=BINARY_LABELS,
    lr=BINARY_LR,
    epochs=BINARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=False,
    n_splits=BINARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(binary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(binary_cv.aggregate_classification_report).T.round(3))


Fold 1: accuracy=0.500
Fold 2: accuracy=0.500
Fold 3: accuracy=0.650
Fold 4: accuracy=0.644
Fold 5: accuracy=0.588

              precision  recall  f1-score   support
None              0.619   0.397     0.484  4050.000
Relationship      0.556   0.756     0.641  4050.000
accuracy          0.576   0.576     0.576     0.576
macro avg         0.588   0.576     0.562  8100.000
weighted avg      0.588   0.576     0.562  8100.000


## Final results — ternary classifier

Re-run 5-fold CV at the chosen `(TERNARY_LR, TERNARY_EPOCHS)` with inverse-
frequency class weights to produce the final reportable per-fold accuracies and
pooled classification report.


In [16]:
TERNARY_LR = 0.01
TERNARY_EPOCHS = 50
TERNARY_FOLDS = 5

In [ ]:
ternary_cv = cross_validate(
    ternary_data,
    embsize=embsize,
    label_map=TERNARY_LABELS,
    lr=TERNARY_LR,
    epochs=TERNARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=True,
    n_splits=TERNARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(ternary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(ternary_cv.aggregate_classification_report).T.round(3))


## Save final results

Pickle a `{"binary": CrossValidationResult, "ternary": CrossValidationResult}`
dict to `CV_RESULTS_PATH` for downstream analysis notebooks to load.


In [15]:
CV_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "binary": binary_cv,
    "ternary": ternary_cv,
    "embeddings_path": str(EMBEDDINGS_PATH),
    "gene_subset_path": str(GENE_SUBSET_PATH) if GENE_SUBSET_PATH else None,
}
with open(CV_RESULTS_PATH, "wb") as f:
    pickle.dump(payload, f)

print(f"Saved CV results to {CV_RESULTS_PATH}")
print(f"  binary:  lr={binary_cv.config['lr']}, epochs={binary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(binary_cv.fold_accuracies):.3f}")
print(f"  ternary: lr={ternary_cv.config['lr']}, epochs={ternary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(ternary_cv.fold_accuracies):.3f}")


Saved CV results to ../data/scgpt_cv_results_cd20.pkl
  binary:  lr=0.001, epochs=20, mean fold acc=0.576
  ternary: lr=1e-05, epochs=20, mean fold acc=0.371
